<a href="https://colab.research.google.com/github/pachterlab/varseek-examples/blob/main/vk_denovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [vk denovo](https://github.com/pachterlab/varseek) demonstration
Call de novo variants from scRNA-seq reads, build a varseek reference from those variants, and count variant-supporting reads with vk count. This notebook uses a [10x PBMC 1k dataset](https://www.10xgenomics.com/datasets/1-k-pbm-cs-from-a-healthy-donor-v-3-chemistry-3-standard-3-0-0) as an example.

Written by Joseph Rich.
___


### Install varseek, and import all packages

In [1]:
try:
    import varseek as vk
except ImportError:
    print("varseek not found, installing...")
    !pip install -U -q varseek

In [2]:
import os
import pandas as pd

import varseek as vk

### Define important paths

In [3]:
# input files
fastqs_dir = os.path.join("data", "pbmc_1k_v3_fastqs")
technology = "10xv3"
reference_dir = os.path.join("data", "reference")
sequences = os.path.join(reference_dir, "Homo_sapiens.GRCh38.cdna.all.fa")

# vk denovo out
variants_dir = os.path.join("data", "pbmc_1k_v3_variants")
variants_vcf = os.path.join(variants_dir, "variants.vcf.gz")
variants = os.path.join(variants_dir, "variants.tsv")
denovo_bam_dir = os.path.join(variants_dir, "bams")
denovo_bowtie2_index_prefix = os.path.join(reference_dir, "Homo_sapiens.GRCh38.cdna.all")
denovo_bowtie2_alignment_dir = os.path.join(variants_dir, "bowtie2_alignments")

# vk ref out
vk_ref_out_dir = os.path.join("data", "varseek_ref_out_pbmc_1k_v3")
vcrs_index = os.path.join(vk_ref_out_dir, "vcrs_index_denovo.idx")
vcrs_t2g = os.path.join(vk_ref_out_dir, "vcrs_t2g_denovo.txt")

# vk count out
vk_count_out_dir = os.path.join("data", "varseek_count_out_pbmc_1k_v3")

# general parameters
w = 37
k = 41
min_counts_denovo = 3
threads = 16

### Download the PBMC fastq dataset

In [4]:
if not os.path.exists(fastqs_dir) or len(os.listdir(fastqs_dir)) == 0:
    !mkdir -p data && \
        cd data && \
        curl -O https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_fastqs.tar && \
        tar -xvf pbmc_1k_v3_fastqs.tar && \
        rm pbmc_1k_v3_fastqs.tar

### Download the reference genome (GRCh38, Ensembl 114, cDNA file)

In [5]:
if not os.path.exists(sequences):
    !gget ref -w cdna -r 114 --out_dir {reference_dir} -d human
    !gunzip {sequences}.gz

### Run varseek denovo

In [6]:
if not os.path.exists(variants):
    fastq_r2_files = sorted(
        os.path.join(root, file)
        for root, _, files in os.walk(fastqs_dir)
        for file in files
        if file.endswith((".fastq.gz", ".fq.gz", ".fastq", ".fq")) and "_R2_" in file
    )

    vk.denovo(
        inputs=fastq_r2_files,
        fasta_ref=sequences,
        aligner="bowtie2",
        bowtie2_genome_index_prefix=denovo_bowtie2_index_prefix,
        bowtie2_alignment_dir=denovo_bowtie2_alignment_dir,
        out_bam_dir=denovo_bam_dir,
        output=variants_vcf,
        output_tsv=variants,
        tsv_reference_type="cdna",
        min_counts=min_counts_denovo,
        threads=threads,
        verbose=1,
    )

variants_df = pd.read_csv(variants, sep="\t")
print(f"Number of variants: {len(variants_df)}")
variants_df.head()

Number of variants: 28377


,seq_id,variant
0,ENST00000390477.2,c.599C>A
1,ENST00000390477.2,c.714_715insGAAG
2,ENST00000390477.2,c.715_716insAAGGAT
3,ENST00000390477.2,c.716_717insAGGATAT
4,ENST00000611116.2,c.584C>A


### Run varseek ref

In [9]:
if not os.path.exists(vcrs_index):
    vk_ref_output_dict = vk.ref(
        variants=variants,
        sequences=sequences,
        seq_id_column="seq_id",
        var_column="variant",
        out=vk_ref_out_dir,
        reference_out_dir=reference_dir,
        dlist_reference_source="t2t",
        index_out=vcrs_index,
        t2g_out=vcrs_t2g,
        w=w,
        k=k
    )

### Run varseek count

This will run the following commands:
- `varseek fastqpp`: Preprocess the fastq files. By default, does nothing.
- `kb count` (variant reference): Perform variant screening on fastq data utilizing kb count's pseudoalignment algorithm. Variant data is stored in an Anndata object as as cell/sample x variant matrix.
- `kb count` ("normal" reference genome) (optional): Perform pseudoalignment of fastq data to the reference genome. Only performed if utilized in the subsequence "varseek clean" step. By default, will occur if the path to the necessary files are not provided as input.
- `varseek clean`: Process the output of kb count. By default, this will threshold variant counts and ensure that there is agreement for each read between the gene of the variant to which the read aligned during the variant reference pseudoalignment and the gene to which the read aligned during the "normal" reference genome pseudoalignment.
- `varseek summarize`: Produces a text file summarizing some high-level insights from the variant screening process.

In [14]:
vk_count_output_dict = vk.count(
    fastqs_dir,
    index=vcrs_index,
    t2g=vcrs_t2g,
    technology=technology,
    out=vk_count_out_dir,
    k=k,
    threads=threads,
)

15:45:56 - INFO - Removing index files from fastq files list, as they are not utilized in kb count with technology 10XV3
15:45:56 - INFO - Setting length_required to 41 if fastqpp is run
15:45:56 - INFO - Skipping vk fastqpp because there was no use for it
15:45:56 - INFO - Skipping kb count because file data/varseek_count_out_pbmc_1k_v3/kb_count_out_vcrs/counts_unfiltered/adata.h5ad already exists and overwrite=False
15:45:56 - INFO - Skipping kb count for reference genome because the reference genome adata object was not needed and/or the file 'data/varseek_count_out_pbmc_1k_v3/kb_count_out_reference_genome/counts_unfiltered/adata.h5ad' already exists. Note that even setting overwrite=True will still not overwrite this particular file.
15:45:56 - INFO - Skipping vk clean because file data/varseek_count_out_pbmc_1k_v3/adata_cleaned.h5ad already exists and overwrite=False
15:45:56 - INFO - Skipping vk summarize because summarize=False
15:45:56 - INFO - Total runtime for vk count: 0m, 0

In [16]:
print(f"Find the processed adata object for further analysis in {vk_count_output_dict['adata_path']}")

Find the processed adata object for further analysis in /home/jrich/Desktop/varseek-examples/data/varseek_count_out_pbmc_1k_v3/adata_cleaned.h5ad
